In [0]:
%sql

CREATE TABLE medallion_project.taxi_data.`01_bronze` AS SELECT * FROM read_files(
    '/Volumes/medallion_project/taxi_data/00_landing_data/',
    format => 'csv') LIMIT 0;


In [0]:
%sql 

COPY INTO medallion_project.taxi_data.`01_bronze`
FROM '/Volumes/medallion_project/taxi_data/00_landing_data/'
FILEFORMAT = CSV
FORMAT_OPTIONS ('delimiter' = ',', 'header' = 'true', 'inferSchema' = 'true');

In [0]:
files_to_move = [
    'yellow_tripdata_2019-07.csv',
    'yellow_tripdata_2019-08.csv',
    'yellow_tripdata_2019-09.csv'
]

source_path = '/Volumes/medallion_project/taxi_data/00_landing_data/'
dest_path = '/Volumes/medallion_project/taxi_data/00_landing_data/raw_q3/'

for file in files_to_move:
    dbutils.fs.mv(source_path + file, dest_path + file)

In [0]:
%sql
CREATE SCHEMA medallion_project.01_bronze;

In [0]:

%sql
CREATE TABLE medallion_project.bronze01.trips_raw DEEP CLONE medallion_project.taxi_data.`01_bronze`;



In [0]:
%sql
DROP TABLE medallion_project.taxi_data.`01_bronze`;

# Configure Auto Loader

In [0]:
(spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("cloudFiles.schemaLocation", "/Volumes/medallion_project/taxi_data/00_landing_data/raw_q3/_schemas/")
  .option("header", "true")
  .option("delimiter", ",")
  .load("/Volumes/medallion_project/taxi_data/00_landing_data/raw_q3/")
  .writeStream
  .option("checkpointLocation", "/Volumes/medallion_project/taxi_data/00_landing_data/raw_q3/_checkpoints/")
  .trigger(availableNow=True)
  .toTable("medallion_project.01_bronze.trips_with_al_raw")
)